# Customer Sentiment & NLP Analysis

**Business problem:** A product team wants to move beyond star ratings and quantify *what* customers praise or complain about in written reviews — which themes drive positive vs negative sentiment, and where the biggest complaint volume sits by product category.

**Approach in this notebook:**
1. Explore ~5,000 customer reviews (EDA: rating mix, sentiment by category, review length)
2. Normalize review text
3. Train two scikit-learn text classifiers (TF-IDF + Logistic Regression vs Naive Bayes) on sentiment labels derived from star ratings
4. Extract the most positive / negative n-grams as "voice of customer" drivers

> **Data note:** all reviews are **synthetic** — generated by `src/generate_data.py` (seed 42). No real customers, products, or brands. See `data/README.md`.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path("../src").resolve()))
from data_cleaning import add_sentiment_label, clean_text

ROOT = Path("..").resolve()
IMG = ROOT / "images"
IMG.mkdir(exist_ok=True)
DPI = 150

df = pd.read_csv(ROOT / "data" / "reviews.csv")
df["sentiment"] = df["rating"].map(add_sentiment_label)
print(f"{len(df):,} reviews | {df['product_category'].nunique()} categories | "
      f"{df['review_date'].min()} to {df['review_date'].max()}")
df.head(3)

5,000 reviews | 5 categories | 2024-01-01 to 2025-12-31


,review_id,review_date,product_category,rating,review_text,sentiment
0,1,2024-03-30,headphones,5,The noise cancellation exceeded my expectation...,positive
1,2,2024-06-08,headphones,4,Stopped working after two days. The sound qual...,positive
2,3,2024-02-14,running shoes,3,Acceptable running shoes. The heel grip could ...,neutral


## 1. Exploratory data analysis

First, the rating mix and how sentiment breaks down by product category.

In [2]:
order = ["positive", "neutral", "negative"]
share = df.groupby(["product_category", "sentiment"]).size().unstack(fill_value=0)
share = share.div(share.sum(axis=1), axis=0).mul(100)[order]

fig, ax = plt.subplots(figsize=(9, 4.5))
share.plot(kind="bar", stacked=True, ax=ax,
           color=["#2ca02c", "#ffbf00", "#d62728"])
ax.set_title("Sentiment share by product category (derived from star ratings)")
ax.set_ylabel("% of reviews")
ax.set_xlabel("")
ax.legend(title="sentiment", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(IMG / "sentiment_distribution.png", dpi=DPI)
plt.close(fig)
print("saved images/sentiment_distribution.png")
share.round(1)

saved images/sentiment_distribution.png


sentiment,positive,neutral,negative
product_category,,,
blender,68.5,15.2,16.3
coffee maker,68.3,15.2,16.5
headphones,67.5,15.5,17.0
running shoes,67.4,15.6,17.0
smartwatch,67.2,14.0,18.8


Do unhappy customers write longer reviews? Complaint text tends to run long, so let's check word count vs star rating.

In [3]:
df["word_count"] = df["review_text"].str.split().str.len()
by_rating = df.groupby("rating")["word_count"].agg(["mean", "median", "count"]).round(1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_rating.index.astype(str), by_rating["mean"], color="#1f77b4")
ax.set_title("Average review length (words) by star rating")
ax.set_xlabel("star rating")
ax.set_ylabel("avg words per review")
for i, v in enumerate(by_rating["mean"]):
    ax.text(i, v + 0.15, f"{v:.1f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(IMG / "rating_vs_length.png", dpi=DPI)
plt.close(fig)
print("saved images/rating_vs_length.png")
by_rating

saved images/rating_vs_length.png


,mean,median,count
rating,,,
1,23.3,23.0,483
2,22.8,22.0,374
3,12.8,12.0,755
4,13.6,13.0,1484
5,13.7,13.0,1904


## 2. Text cleaning

Lowercase → strip URLs → keep letters/spaces → collapse whitespace → drop English stopwords (scikit-learn's built-in list, no extra downloads).

In [4]:
df["clean_text"] = df["review_text"].map(clean_text)
sample = df.sample(2, random_state=7)[["rating", "review_text", "clean_text"]]
for _, r in sample.iterrows():
    print(f"[{r['rating']}★] raw:   {r['review_text']}")
    print(f"       clean: {r['clean_text']}\n")

[2★] raw:   Stopped working after two days. The watch band is completely difficult. Save your money and look elsewhere. I contacted support and they were completely scratchy.
       clean: stopped working days watch band completely difficult save money look contacted support completely scratchy

[4★] raw:   Super sturdy and the tread is durable. I enjoy buying this.
       clean: super sturdy tread durable enjoy buying



## 3. Sentiment classification

Labels come from star ratings (1–2 negative, 3 neutral, 4–5 positive). Two TF-IDF baselines, stratified 80/20 split, `random_state=42`:
- **TF-IDF (1–2 grams) + Logistic Regression**
- **TF-IDF (1–2 grams) + Multinomial Naive Bayes**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

LABELS = ["positive", "neutral", "negative"]
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"])
print(f"train={len(X_train):,}  test={len(X_test):,}")

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9)
models = {
    "LogisticRegression": Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=1000, random_state=42))]),
    "MultinomialNB": Pipeline([("tfidf", tfidf), ("clf", MultinomialNB())]),
}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"\n{'='*55}\n{name}: accuracy={accuracy_score(y_test, pred):.4f}\n{'='*55}")
    print(classification_report(y_test, pred, labels=LABELS, digits=3))

train=4,000  test=1,000



LogisticRegression: accuracy=0.9080
              precision    recall  f1-score   support

    positive      0.951     0.925     0.938       678
     neutral      1.000     0.894     0.944       151
    negative      0.709     0.854     0.775       171

    accuracy                          0.908      1000
   macro avg      0.887     0.891     0.886      1000
weighted avg      0.917     0.908     0.911      1000




MultinomialNB: accuracy=0.8930
              precision    recall  f1-score   support

    positive      0.937     0.917     0.927       678
     neutral      1.000     0.768     0.869       151
    negative      0.705     0.906     0.793       171

    accuracy                          0.893      1000
   macro avg      0.880     0.864     0.863      1000
weighted avg      0.907     0.893     0.895      1000



In [6]:
# Confusion matrix for the Logistic Regression model
lr = models["LogisticRegression"]
cm = confusion_matrix(y_test, lr.predict(X_test), labels=LABELS)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues", ax=ax,
            xticklabels=LABELS, yticklabels=LABELS)
ax.set_title("Confusion matrix — Logistic Regression (% of true class)")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
fig.tight_layout()
fig.savefig(IMG / "confusion_matrix.png", dpi=DPI)
plt.close(fig)
print("saved images/confusion_matrix.png")
pd.DataFrame(cm, index=LABELS, columns=LABELS)

saved images/confusion_matrix.png


,positive,neutral,negative
positive,627,0,51
neutral,7,135,9
negative,25,0,146


## 4. What drives sentiment? Top n-grams

The logistic regression coefficients tell us which words/phrases push a review toward positive or negative — the "voice of customer" themes a product team would act on.

In [7]:
vec = lr.named_steps["tfidf"]
clf = lr.named_steps["clf"]
terms = np.array(vec.get_feature_names_out())
classes = list(clf.classes_)

def _top_drivers(class_name, n=15):
    coefs = clf.coef_[classes.index(class_name)]
    order = np.argsort(coefs)[::-1]
    return pd.DataFrame({"ngram": terms[order[:n]], "weight": coefs[order[:n]].round(3)})

top_pos = _top_drivers("positive")
top_neg = _top_drivers("negative")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
axes[0].barh(top_pos["ngram"][::-1], top_pos["weight"][::-1], color="#2ca02c")
axes[0].set_title("Top positive n-grams")
axes[1].barh(top_neg["ngram"], top_neg["weight"], color="#d62728")
axes[1].set_title("Top negative n-grams")
for ax in axes:
    ax.set_xlabel("logistic regression weight")
fig.suptitle("Strongest sentiment drivers (TF-IDF + Logistic Regression)")
fig.tight_layout()
fig.savefig(IMG / "top_ngrams.png", dpi=DPI)
plt.close(fig)
print("saved images/top_ngrams.png")
print("TOP POSITIVE"); print(top_pos.to_string(index=False))
print("\nTOP NEGATIVE"); print(top_neg.to_string(index=False))

saved images/top_ngrams.png
TOP POSITIVE
                ngram  weight
           just works   2.245
               bought   2.210
                works   1.883
              happier   1.804
                month   1.804
                 just   1.657
             exceeded   1.349
exceeded expectations   1.349
         expectations   1.349
          performance   1.349
          money spent   1.347
                spent   1.347
                  day   1.278
                setup   1.278
              use day   1.278

TOP NEGATIVE
          ngram  weight
       returned   1.834
            buy   1.709
        arrived   1.395
         design   1.297
    makes worse   1.282
          makes   1.282
          worse   1.282
         blurry   1.226
        leaking   1.209
        working   1.110
        stopped   1.110
stopped working   1.110
       assembly   1.101
           star   1.101
       terrible   1.095


## 5. Key insights

*(Numbers below come from the executed run above.)*

- **Rating skew is real:** ~two-thirds of reviews are 4–5 stars; the negative class is the minority, which is why per-class recall matters more than raw accuracy here.
- **Complaints run longer:** 1-star reviews average noticeably more words than 5-star reviews — unhappy customers explain themselves.
- **Linear models are strong baselines:** TF-IDF + Logistic Regression separates positive/negative cleanly; most confusion sits around the small neutral class (3-star reviews mix mild praise with mild criticism).
- **Drivers are interpretable:** positive weights cluster on quality/durability language (*excellent, amazing, outstanding, reliable*), negative weights on defect language (*broken, defective, terrible, awful, waste*).

## Extensions (not done here)

- Fine-tune a transformer (e.g. BERT/RoBERTa) and compare against these TF-IDF baselines — the honest next experiment, especially for the ambiguous neutral class.
- Aspect-based sentiment: attribute praise/complaints to specific product features per category.
- Time-trend the % negative KPI per category for an early-warning dashboard.